In [1]:
import ktrain
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from ktrain import text

In [2]:
DATA_PATH = r'train.csv'
NUM_WORDS = 50000
MAXLEN = 264

In [3]:
(x_train, y_train), (x_test, y_test), preproc = text.texts_from_csv(DATA_PATH,
                      'Text',
                      label_columns = ['STRENGTHS','WEAKNESSES','OPPORTUNITIES','THREATS'],
                      preprocess_mode='bert',
                      val_filepath=None, 
                      max_features=NUM_WORDS,
                      maxlen=MAXLEN)

detected encoding: ISO-8859-3 (if wrong, set manually)
['STRENGTHS', 'WEAKNESSES', 'OPPORTUNITIES', 'THREATS']
      STRENGTHS  WEAKNESSES  OPPORTUNITIES  THREATS
1174          0           1              0        0
62            1           0              0        0
1187          0           1              0        0
2071          0           0              0        1
495           1           0              0        0
['STRENGTHS', 'WEAKNESSES', 'OPPORTUNITIES', 'THREATS']
      STRENGTHS  WEAKNESSES  OPPORTUNITIES  THREATS
1941          0           0              1        0
739           1           0              0        0
1828          0           0              1        0
1342          0           1              0        0
1263          0           1              0        0
downloading pretrained BERT model (uncased_L-12_H-768_A-12.zip)...
[██████████████████████████████████████████████████]
extracting pretrained BERT model...
done.

cleanup downloaded zip...
done.

preprocessing

Is Multi-Label? False
preprocessing test...
language: en


In [5]:
x_test[:5]

[array([[  101,  2582,  1010, ...,     0,     0,     0],
        [  101,  4591,  1998, ...,     0,     0,     0],
        [  101, 13970, 17421, ...,     0,     0,     0],
        ...,
        [  101,  1996,  2796, ...,     0,     0,     0],
        [  101,  1996,  3754, ...,     0,     0,     0],
        [  101,  1996,  2194, ...,     0,     0,     0]]),
 array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]])]

In [6]:
y_test[:5]

array([[0., 0., 1., 0.],
       [1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.]], dtype=float32)

In [7]:
model = text.text_classifier('bert', train_data=(x_train, y_train), preproc=preproc)

Is Multi-Label? False
maxlen is 264
done.


In [8]:
learner = ktrain.get_learner(model, train_data=(x_train, y_train), batch_size=16)

In [ ]:
learner.fit_onecycle(2e-5, 4)



begin training using onecycle policy with max lr of 2e-05...
Epoch 1/4
  3/141 [..............................] - ETA: 1:33:26 - loss: 1.7687 - accuracy: 0.1458

In [ ]:
learner.validate(val_data=(x_test, y_test), class_names=['S','W','O','T'])

In [ ]:
predictor = ktrain.get_predictor(learner.model, preproc)

In [ ]:
predictor.get_classes()

In [ ]:
df1 = pd.read_excel(r"test.xlsm",index=1)

In [ ]:
df1.set_index("#id", inplace = True)

In [ ]:
display(HTML(df1[15:16].to_html()))

In [ ]:
predictor.predict(np.array(df1['Text'][15:16]))

In [ ]:
# let's save the predictor for later use
predictor.save(r'.\run264_16\saved_model_25k')

In [ ]:
# reload the predictor
reloaded_predictor = ktrain.load_predictor(r'.\run264_16\saved_model_25k')

In [ ]:
# make a prediction on the same document to verify it still works
results=reloaded_predictor.predict(np.array(df1['Text'][0:16]))

In [ ]:
df1['Prediction']=results
display(HTML(df1.to_html()))